# 05 — Run the trains

Each trip becomes a path through the drawn network plus keyframes saying how far
along it the train is at a given second. The browser interpolates: it brackets
the current time between two keyframes, lerps a distance, and asks the SVG path
for the point there.

In [ ]:
%load_ext autoreload
%autoreload 2

from schematic import feeds, loom, pipeline, animate
from schematic.linegraph import LineGraph
from schematic.crs import to_mercator
from schematic.render import render, octilinearity, Style

FEED = "la-metro-rail"
LINE_ORDER = list("ABCDEK")   # the order lines are drawn in, back to front

In [ ]:
result = pipeline.run(FEED, line_order=LINE_ORDER)
print(result.summary())

### The service day

Times past midnight stay past midnight — `25:44:00` is 1:44am, not 1:44am
yesterday. Wrapping them would teleport the last trains of the night back to the
start of the day.

In [ ]:
from schematic.schedule import format_gtfs_time, concurrent_trips

trips = result.trips
print(f"{len(trips)} trips on {result.date}")
print("first departure", format_gtfs_time(min(t.start for t in trips)))
print("last arrival   ", format_gtfs_time(max(t.end for t in trips)))
for h in range(4, 26, 2):
    n = len(concurrent_trips(trips, h * 3600))
    print(f"  {format_gtfs_time(h*3600)}  {'#' * n} {n}")

### Path deduplication

A whole day of service collapses to a handful of stopping patterns, so the
payload carries a few dozen paths rather than a few thousand.

In [ ]:
anim = result.animation
print(f"{len(anim.trips)} trips -> {len(anim.paths)} distinct paths")
for i, p in enumerate(anim.paths[:8]):
    print(f"  path {i}: line {p['route']}, {len(p['stops'])} stops, {len(p['d'])} chars")

### Check a frame before trusting the browser

Positions computed here, from the emitted payload alone, using the same maths
the JavaScript does. If these land on the lines, the animation will too.

In [ ]:
import bisect
from IPython.display import SVG, display
from schematic.offsets import point_at

def length_at(keys, t):
    if t <= keys[0][0]: return keys[0][1]
    if t >= keys[-1][0]: return keys[-1][1]
    i = bisect.bisect_right([k[0] for k in keys], t) - 1
    (ta, la), (tb, lb) = keys[i], keys[i+1]
    return lb if tb == ta else la + (lb - la) * (t - ta) / (tb - ta)

geoms = [[tuple(map(float, tok.split())) for tok in p["d"].replace("M", " ").split("L") if tok.strip()]
         for p in anim.paths]

T = 8 * 3600
dots = [(point_at(geoms[tr["p"]], length_at(tr["k"], T)), anim.lines.get(tr["r"], "#333"))
        for tr in anim.trips if tr["k"][0][0] <= T <= tr["k"][-1][0]]
print(f"{len(dots)} trains at {format_gtfs_time(T)}")

svg = result.render.svg.replace("</svg>", "".join(
    f'<circle cx="{x:.1f}" cy="{y:.1f}" r="6" fill="{c}" stroke="#fff" stroke-width="2"/>'
    for (x, y), c in dots) + "</svg>")
display(SVG(svg))

### The interactive version

`out/animation.html` is self-contained — open it in a browser.

In [ ]:
from pathlib import Path
for f in sorted(Path("out").glob("*")):
    if f.suffix in {".html", ".json", ".svg"}:
        print(f"{f.name:24} {f.stat().st_size/1024:8.0f} KB")